In [ ]:
# Download checkpoint from Hugging Face
from huggingface_hub import snapshot_download, login
import os

# Login with token (paste your token here)
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = ""  # <-- Replace with your actual token
login(token=HF_TOKEN)

checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Download all checkpoints from the repository
snapshot_download(
    repo_id="",
    local_dir=checkpoint_dir,
)

print(f"Checkpoints downloaded to: {checkpoint_dir}")
print("Files:", os.listdir(checkpoint_dir))

Fetching 32 files: 100%|██████████| 32/32 [00:13<00:00,  2.39it/s]

Checkpoints downloaded to: ./checkpoints
Files: ['.cache', '.gitattributes', 'neurogs_ckpt_iter1000.pt', 'neurogs_ckpt_iter10000.pt', 'neurogs_ckpt_iter13000.pt', 'neurogs_ckpt_iter15000.pt', 'neurogs_ckpt_iter14000.pt', 'neurogs_ckpt_iter11000.pt', 'neurogs_ckpt_iter12000.pt', 'neurogs_ckpt_iter16000.pt', 'neurogs_ckpt_iter17000.pt', 'neurogs_ckpt_iter18000.pt', 'neurogs_ckpt_iter2000.pt', 'neurogs_ckpt_iter19000.pt', 'neurogs_ckpt_iter20000.pt', 'neurogs_ckpt_iter22000.pt', 'neurogs_ckpt_iter21000.pt', 'neurogs_ckpt_iter23000.pt', 'neurogs_ckpt_iter3000.pt', 'neurogs_ckpt_iter24000.pt', 'neurogs_ckpt_iter27000.pt', 'neurogs_ckpt_iter25000.pt', 'neurogs_ckpt_iter26000.pt', 'neurogs_ckpt_iter4000.pt', 'neurogs_ckpt_iter28000.pt', 'neurogs_ckpt_iter29000.pt', 'neurogs_ckpt_iter5000.pt', 'neurogs_ckpt_iter30000.pt', 'neurogs_ckpt_iter6000.pt', 'neurogs_ckpt_iter7000.pt', 'neurogs_ckpt_iter8000.pt', 'neurogs_ckpt_iter9000.pt', 'neurogs_codec_ckpt_final.pt']


# Codec Comparison: Classical & Neural Baselines for 3D Volume Compression

This notebook implements comprehensive baseline comparisons for 3D microscopy volume compression:

**Lossless classical codecs:**
- TIFF-LZW / TIFF-Deflate
- HDF5 + gzip
- Zarr + Blosc-zstd

**Lossy classical codecs:**
- JPEG2000 (slice-wise)
- H.265/HEVC (treating z-slices as video frames)

**Neural implicit baselines:**
- SIREN-3D (sinusoidal activation INR)
- PE-MLP (Fourier/Positional Encoding features)

**Implementation notes for fairness:**
- All methods operate on the same normalized input volumes
- Evaluated on the same metrics (PSNR, SSIM, compression ratio)
- Both compressed size (true bytes on disk) and estimated rate reported
- Comparable optimization budgets for neural methods

## 0) Setup & Dependencies

In [5]:
# Install required packages (uncomment as needed)
# !pip install tifffile h5py zarr blosc imageio-ffmpeg glymur opencv-python torch numpy tqdm

In [2]:
import os
import io
import gzip
import math
import time
import tempfile
import subprocess
from dataclasses import dataclass, field
from typing import Tuple, Dict, List, Optional, Any
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# Image I/O
try:
    import tifffile as tiff
except ImportError:
    raise ImportError("Please install tifffile: pip install tifffile")

# HDF5
try:
    import h5py
except ImportError:
    print("Warning: h5py not installed. HDF5 baselines will be skipped.")
    h5py = None

# Zarr + Blosc
try:
    import zarr
    from numcodecs import Blosc
except ImportError:
    print("Warning: zarr/numcodecs not installed. Zarr baselines will be skipped.")
    zarr = None

# JPEG2000
try:
    import glymur
    HAVE_JPEG2000 = True
except ImportError:
    print("Warning: glymur not installed. JPEG2000 baselines will be skipped.")
    HAVE_JPEG2000 = False

# OpenCV for video codec
try:
    import cv2
    HAVE_OPENCV = True
except ImportError:
    print("Warning: OpenCV not installed. H.265/HEVC baselines will be skipped.")
    HAVE_OPENCV = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

/venv/neurogs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
PyTorch version: 2.10.0+cu128


## 1) Load 3D Volume & Configuration

In [7]:
# ---- User Configuration ----
TIF_PATH = "10-2900-control-cell-05_cropped_corrected.tif"  # <-- Set your volume path
VOXEL_SPACING = (0.126, 0.126, 1.0)  # (dx, dy, dz) in microns
OUTPUT_DIR = Path("codec_comparison_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
# -----------------------------

@dataclass
class CompressionResult:
    """Stores results from a single codec evaluation."""
    name: str
    compressed_bytes: int
    original_bytes: int
    compression_ratio: float
    bpp: float  # bits per voxel
    psnr: Optional[float] = None
    ssim: Optional[float] = None
    encode_time: float = 0.0
    decode_time: float = 0.0
    is_lossless: bool = False
    notes: str = ""

# Results storage
results: List[CompressionResult] = []

def load_volume(path: str) -> Tuple[np.ndarray, np.ndarray]:
    """Load 3D volume, return original and normalized [0,1] float32 versions."""
    assert os.path.exists(path), f"File not found: {path}"
    V_np = tiff.imread(path)
    assert V_np.ndim == 3, f"Expected 3D volume, got shape {V_np.shape}"
    
    print(f"Loaded: {V_np.shape}, dtype={V_np.dtype}, min/max={V_np.min()}/{V_np.max()}")
    
    # Store original for lossless comparison
    V_original = V_np.copy()
    
    # Normalize to [0,1] float32
    V_norm = V_np.astype(np.float32)
    V_norm = (V_norm - V_norm.min()) / (V_norm.max() - V_norm.min() + 1e-8)
    
    return V_original, V_norm

V_original, V_norm = load_volume(TIF_PATH)
V_t = torch.from_numpy(V_norm).to(DEVICE)  # GPU tensor

# Calculate original size
original_bytes = V_original.nbytes
print(f"Original volume: {V_original.shape}, {original_bytes:,} bytes ({original_bytes/1e6:.2f} MB)")

Loaded: (100, 647, 813), dtype=uint8, min/max=2/255
Original volume: (100, 647, 813), 52,601,100 bytes (52.60 MB)


## 2) Evaluation Metrics

In [8]:
def compute_psnr(original: np.ndarray, reconstructed: np.ndarray, 
                 data_range: float = None) -> float:
    """Compute Peak Signal-to-Noise Ratio."""
    if data_range is None:
        data_range = original.max() - original.min()
    
    mse = np.mean((original.astype(np.float64) - reconstructed.astype(np.float64)) ** 2)
    if mse < 1e-10:
        return float('inf')
    return float(10.0 * np.log10(data_range ** 2 / mse))

def compute_ssim_3d(original: np.ndarray, reconstructed: np.ndarray, 
                    win_size: int = 7) -> float:
    """Compute 3D SSIM using patch-based approach."""
    # Convert to float and normalize
    a = original.astype(np.float32)
    b = reconstructed.astype(np.float32)
    
    # Normalize to [0,1] for consistent SSIM calculation
    a_min, a_max = a.min(), a.max()
    a = (a - a_min) / (a_max - a_min + 1e-8)
    b = (b - a_min) / (a_max - a_min + 1e-8)  # Use same normalization
    
    # Convert to torch for efficient computation
    a_t = torch.from_numpy(a).to(DEVICE)[None, None]  # (1,1,Z,Y,X)
    b_t = torch.from_numpy(b).to(DEVICE)[None, None]
    
    K1, K2 = 0.01, 0.03
    C1, C2 = K1**2, K2**2
    pad = win_size // 2
    
    mu_a = F.avg_pool3d(a_t, win_size, stride=1, padding=pad)
    mu_b = F.avg_pool3d(b_t, win_size, stride=1, padding=pad)
    
    sigma_a = F.avg_pool3d(a_t * a_t, win_size, stride=1, padding=pad) - mu_a * mu_a
    sigma_b = F.avg_pool3d(b_t * b_t, win_size, stride=1, padding=pad) - mu_b * mu_b
    sigma_ab = F.avg_pool3d(a_t * b_t, win_size, stride=1, padding=pad) - mu_a * mu_b
    
    ssim_map = ((2 * mu_a * mu_b + C1) * (2 * sigma_ab + C2)) / \
               ((mu_a**2 + mu_b**2 + C1) * (sigma_a + sigma_b + C2) + 1e-8)
    
    return float(ssim_map.mean().cpu())

def evaluate_reconstruction(original: np.ndarray, reconstructed: np.ndarray, 
                          is_lossless: bool = False) -> Tuple[float, float]:
    """Evaluate reconstruction quality."""
    if is_lossless:
        # For lossless, verify exact match
        if np.array_equal(original, reconstructed):
            return float('inf'), 1.0
        else:
            print("Warning: Lossless codec did not achieve exact reconstruction!")
    
    psnr = compute_psnr(original, reconstructed)
    ssim = compute_ssim_3d(original, reconstructed)
    return psnr, ssim

def add_result(name: str, compressed_bytes: int, original_bytes: int,
               psnr: float, ssim: float, encode_time: float, decode_time: float,
               is_lossless: bool = False, notes: str = ""):
    """Add a compression result to the global results list."""
    ratio = original_bytes / compressed_bytes if compressed_bytes > 0 else float('inf')
    bpp = (compressed_bytes * 8) / (V_original.size)
    
    result = CompressionResult(
        name=name,
        compressed_bytes=compressed_bytes,
        original_bytes=original_bytes,
        compression_ratio=ratio,
        bpp=bpp,
        psnr=psnr,
        ssim=ssim,
        encode_time=encode_time,
        decode_time=decode_time,
        is_lossless=is_lossless,
        notes=notes
    )
    results.append(result)
    
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"  Compressed: {compressed_bytes:,} bytes ({compressed_bytes/1e6:.2f} MB)")
    print(f"  Ratio: {ratio:.2f}x | BPP: {bpp:.4f}")
    print(f"  PSNR: {psnr:.2f} dB | SSIM: {ssim:.4f}")
    print(f"  Encode: {encode_time:.2f}s | Decode: {decode_time:.2f}s")
    if notes:
        print(f"  Notes: {notes}")
    
    return result

---
# Part A: Lossless Classical Codecs

Standard archival baselines for scientific data preservation.

## 3) TIFF-LZW (Lossless)

In [9]:
def codec_tiff_lzw(volume: np.ndarray, output_path: Path) -> CompressionResult:
    """TIFF with LZW compression - lossless, widely supported."""
    try:
        import imagecodecs
    except ImportError:
        print("Skipping TIFF-LZW: imagecodecs not installed. Run: pip install imagecodecs")
        return None
    
    out_file = output_path / "volume_lzw.tif"
    
    # Encode
    t0 = time.time()
    tiff.imwrite(out_file, volume, compression='lzw')
    encode_time = time.time() - t0
    
    compressed_bytes = out_file.stat().st_size
    
    # Decode
    t0 = time.time()
    V_dec = tiff.imread(out_file)
    decode_time = time.time() - t0
    
    # Evaluate
    psnr, ssim = evaluate_reconstruction(volume, V_dec, is_lossless=True)
    
    return add_result(
        name="TIFF-LZW",
        compressed_bytes=compressed_bytes,
        original_bytes=volume.nbytes,
        psnr=psnr, ssim=ssim,
        encode_time=encode_time,
        decode_time=decode_time,
        is_lossless=True,
        notes="Lempel-Ziv-Welch, universally supported"
    )

codec_tiff_lzw(V_original, OUTPUT_DIR)


TIFF-LZW
  Compressed: 26,311,299 bytes (26.31 MB)
  Ratio: 2.00x | BPP: 4.0016
  PSNR: inf dB | SSIM: 1.0000
  Encode: 0.08s | Decode: 0.06s
  Notes: Lempel-Ziv-Welch, universally supported


CompressionResult(name='TIFF-LZW', compressed_bytes=26311299, original_bytes=52601100, compression_ratio=1.999182936577932, bpp=4.0016347947096165, psnr=inf, ssim=1.0, encode_time=0.07560873031616211, decode_time=0.058690786361694336, is_lossless=True, notes='Lempel-Ziv-Welch, universally supported')

## 4) TIFF-Deflate (Lossless)

In [10]:
def codec_tiff_deflate(volume: np.ndarray, output_path: Path) -> CompressionResult:
    """TIFF with Deflate/zlib compression - lossless, better ratio than LZW."""
    try:
        import imagecodecs
    except ImportError:
        print("Skipping TIFF-Deflate: imagecodecs not installed. Run: pip install imagecodecs")
        return None
    
    out_file = output_path / "volume_deflate.tif"
    
    # Encode
    t0 = time.time()
    tiff.imwrite(out_file, volume, compression='deflate', compressionargs={'level': 9})
    encode_time = time.time() - t0
    
    compressed_bytes = out_file.stat().st_size
    
    # Decode
    t0 = time.time()
    V_dec = tiff.imread(out_file)
    decode_time = time.time() - t0
    
    # Evaluate
    psnr, ssim = evaluate_reconstruction(volume, V_dec, is_lossless=True)
    
    return add_result(
        name="TIFF-Deflate",
        compressed_bytes=compressed_bytes,
        original_bytes=volume.nbytes,
        psnr=psnr, ssim=ssim,
        encode_time=encode_time,
        decode_time=decode_time,
        is_lossless=True,
        notes="Deflate/zlib level 9"
    )

codec_tiff_deflate(V_original, OUTPUT_DIR)


TIFF-Deflate
  Compressed: 24,837,254 bytes (24.84 MB)
  Ratio: 2.12x | BPP: 3.7775
  PSNR: inf dB | SSIM: 1.0000
  Encode: 0.17s | Decode: 0.08s
  Notes: Deflate/zlib level 9


CompressionResult(name='TIFF-Deflate', compressed_bytes=24837254, original_bytes=52601100, compression_ratio=2.117830739259662, bpp=3.7774501293699183, psnr=inf, ssim=1.0, encode_time=0.16971993446350098, decode_time=0.0755457878112793, is_lossless=True, notes='Deflate/zlib level 9')

## 5) HDF5 + gzip (Lossless)

In [11]:
def codec_hdf5_gzip(volume: np.ndarray, output_path: Path) -> CompressionResult:
    """HDF5 with gzip compression - scientific data standard."""
    if h5py is None:
        print("Skipping HDF5: h5py not installed")
        return None
    
    out_file = output_path / "volume.h5"
    
    # Encode
    t0 = time.time()
    with h5py.File(out_file, 'w') as f:
        # Use chunking for better compression of 3D data
        chunks = tuple(min(64, s) for s in volume.shape)
        f.create_dataset('volume', data=volume, compression='gzip', 
                        compression_opts=9, chunks=chunks)
    encode_time = time.time() - t0
    
    compressed_bytes = out_file.stat().st_size
    
    # Decode
    t0 = time.time()
    with h5py.File(out_file, 'r') as f:
        V_dec = f['volume'][:]
    decode_time = time.time() - t0
    
    # Evaluate
    psnr, ssim = evaluate_reconstruction(volume, V_dec, is_lossless=True)
    
    return add_result(
        name="HDF5+gzip",
        compressed_bytes=compressed_bytes,
        original_bytes=volume.nbytes,
        psnr=psnr, ssim=ssim,
        encode_time=encode_time,
        decode_time=decode_time,
        is_lossless=True,
        notes="HDF5 chunked (64³) + gzip level 9"
    )

codec_hdf5_gzip(V_original, OUTPUT_DIR)


HDF5+gzip
  Compressed: 22,991,514 bytes (22.99 MB)
  Ratio: 2.29x | BPP: 3.4967
  PSNR: inf dB | SSIM: 1.0000
  Encode: 26.25s | Decode: 0.37s
  Notes: HDF5 chunked (64³) + gzip level 9


CompressionResult(name='HDF5+gzip', compressed_bytes=22991514, original_bytes=52601100, compression_ratio=2.287848464437792, bpp=3.496735087289049, psnr=inf, ssim=1.0, encode_time=26.248629331588745, decode_time=0.3704814910888672, is_lossless=True, notes='HDF5 chunked (64³) + gzip level 9')

## 6) Zarr + Blosc-zstd (Lossless)

In [12]:
def codec_zarr_blosc_zstd(volume: np.ndarray, output_path: Path) -> CompressionResult:
    """Zarr with Blosc-zstd compression - modern chunked array format."""
    if zarr is None:
        print("Skipping Zarr: zarr/numcodecs not installed")
        return None
    
    out_dir = output_path / "volume.zarr"
    
    # Clean up if exists
    import shutil
    if out_dir.exists():
        shutil.rmtree(out_dir)
    
    # Encode with Blosc+zstd (high compression ratio, fast)
    t0 = time.time()
    compressor = Blosc(cname='zstd', clevel=9, shuffle=Blosc.BITSHUFFLE)
    chunks = tuple(min(64, s) for s in volume.shape)
    z = zarr.open(str(out_dir), mode='w', shape=volume.shape, dtype=volume.dtype,
                  chunks=chunks, compressor=compressor)
    z[:] = volume
    encode_time = time.time() - t0
    
    # Calculate compressed size (sum of all chunk files)
    def get_dir_size(path):
        total = 0
        for dirpath, dirnames, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                total += os.path.getsize(fp)
        return total
    
    compressed_bytes = get_dir_size(out_dir)
    
    # Decode
    t0 = time.time()
    z = zarr.open(str(out_dir), mode='r')
    V_dec = z[:]
    decode_time = time.time() - t0
    
    # Evaluate
    psnr, ssim = evaluate_reconstruction(volume, V_dec, is_lossless=True)
    
    return add_result(
        name="Zarr+Blosc-zstd",
        compressed_bytes=compressed_bytes,
        original_bytes=volume.nbytes,
        psnr=psnr, ssim=ssim,
        encode_time=encode_time,
        decode_time=decode_time,
        is_lossless=True,
        notes="Blosc zstd level 9 + bitshuffle, chunked (64³)"
    )

codec_zarr_blosc_zstd(V_original, OUTPUT_DIR)


Zarr+Blosc-zstd
  Compressed: 28,148,583 bytes (28.15 MB)
  Ratio: 1.87x | BPP: 4.2811
  PSNR: inf dB | SSIM: 1.0000
  Encode: 23.52s | Decode: 0.19s
  Notes: Blosc zstd level 9 + bitshuffle, chunked (64³)


CompressionResult(name='Zarr+Blosc-zstd', compressed_bytes=28148583, original_bytes=52601100, compression_ratio=1.8686944206036944, bpp=4.281063780035018, psnr=inf, ssim=1.0, encode_time=23.52121376991272, decode_time=0.18776965141296387, is_lossless=True, notes='Blosc zstd level 9 + bitshuffle, chunked (64³)')

---
# Part B: Lossy Classical Codecs

For 2D codecs, we compress each axial slice independently and reconstruct the full volume by stacking slices.

## 7) JPEG2000 Slice-wise (Lossy)

In [1]:
def codec_jpeg2000_slicewise(volume: np.ndarray, output_path: Path, 
                             quality_layers: List[int] = [50, 100, 200]) -> List[CompressionResult]:
    """
    JPEG2000 slice-wise compression at multiple quality levels.
    Each z-slice is compressed independently, then stacked to reconstruct.
    
    Args:
        quality_layers: PSNR targets in dB (higher = better quality, larger file)
    """
    if not HAVE_JPEG2000:
        print("Skipping JPEG2000: glymur not installed")
        return []
    
    results_j2k = []
    Z, Y, X = volume.shape
    
    # Normalize to 8-bit or 16-bit depending on original dtype
    if volume.dtype == np.uint16:
        v_scaled = volume
        bit_depth = 16
    else:
        # Convert to 16-bit for better quality preservation
        v_min, v_max = volume.min(), volume.max()
        v_scaled = ((volume.astype(np.float64) - v_min) / (v_max - v_min + 1e-10) * 65535).astype(np.uint16)
        bit_depth = 16
    
    for psnr_target in quality_layers:
        out_dir = output_path / f"jpeg2000_psnr{psnr_target}"
        out_dir.mkdir(exist_ok=True)
        
        # Encode all slices
        t0 = time.time()
        total_bytes = 0
        
        for z_idx in range(Z):
            slice_data = v_scaled[z_idx]
            out_file = out_dir / f"slice_{z_idx:04d}.jp2"
            
            # Use cratios (compression ratio) for rate control
            # Higher PSNR target = lower compression ratio
            cratio = max(2, 1000 // psnr_target)  # Approximate mapping
            glymur.Jp2k(str(out_file), slice_data, cratios=[cratio])
            total_bytes += out_file.stat().st_size
        
        encode_time = time.time() - t0
        
        # Decode and reconstruct
        t0 = time.time()
        V_dec = np.zeros_like(v_scaled)
        for z_idx in range(Z):
            out_file = out_dir / f"slice_{z_idx:04d}.jp2"
            jp2 = glymur.Jp2k(str(out_file))
            V_dec[z_idx] = jp2[:]
        decode_time = time.time() - t0
        
        # Convert back to original scale if needed
        if volume.dtype != np.uint16:
            V_dec_orig = (V_dec.astype(np.float64) / 65535 * (v_max - v_min) + v_min).astype(volume.dtype)
        else:
            V_dec_orig = V_dec
        
        # Evaluate
        psnr, ssim = evaluate_reconstruction(volume, V_dec_orig)
        
        result = add_result(
            name=f"JPEG2000 (PSNR≈{psnr_target})",
            compressed_bytes=total_bytes,
            original_bytes=volume.nbytes,
            psnr=psnr, ssim=ssim,
            encode_time=encode_time,
            decode_time=decode_time,
            is_lossless=False,
            notes=f"Slice-wise, {Z} slices, cratio≈{cratio}"
        )
        results_j2k.append(result)
    
    return results_j2k

# Run at multiple quality levels
try:
    codec_jpeg2000_slicewise(V_original, OUTPUT_DIR, quality_layers=[40, 80, 150])
except Exception as e:
    print(f"JPEG2000 codec error: {e}")

NameError: name 'np' is not defined

## 8) H.265/HEVC Video Codec (Lossy)

Treats z-slices as video frames for temporal/spatial redundancy exploitation.

In [14]:
def check_ffmpeg_hevc() -> bool:
    """Check if FFmpeg with HEVC support is available."""
    try:
        result = subprocess.run(['ffmpeg', '-encoders'], capture_output=True, text=True)
        return 'libx265' in result.stdout or 'hevc' in result.stdout
    except FileNotFoundError:
        return False

def codec_hevc_video(volume: np.ndarray, output_path: Path, 
                     crf_values: List[int] = [18, 23, 28, 35]) -> List[CompressionResult]:
    """
    H.265/HEVC codec treating z-slices as video frames.
    Uses FFmpeg with libx265 encoder.
    
    Args:
        crf_values: Constant Rate Factor (0=lossless, 51=worst). Lower = better quality.
    """
    if not check_ffmpeg_hevc():
        print("Skipping HEVC: FFmpeg with libx265 not available")
        return []
    
    results_hevc = []
    Z, Y, X = volume.shape
    
    # Normalize to 8-bit grayscale for video codec
    v_min, v_max = float(volume.min()), float(volume.max())
    v_8bit = ((volume.astype(np.float64) - v_min) / (v_max - v_min + 1e-10) * 255).astype(np.uint8)
    
    for crf in crf_values:
        out_file = output_path / f"volume_hevc_crf{crf}.mp4"
        raw_file = output_path / "temp_raw.yuv"
        
        # Write frames as raw Y (grayscale) format
        t0 = time.time()
        
        # Create raw video file
        with open(raw_file, 'wb') as f:
            for z_idx in range(Z):
                f.write(v_8bit[z_idx].tobytes())
        
        # Encode with FFmpeg
        cmd_encode = [
            'ffmpeg', '-y',
            '-f', 'rawvideo',
            '-pix_fmt', 'gray',
            '-s', f'{X}x{Y}',
            '-r', '1',  # 1 fps (doesn't matter for size)
            '-i', str(raw_file),
            '-c:v', 'libx265',
            '-crf', str(crf),
            '-preset', 'medium',
            '-pix_fmt', 'yuv420p',  # Required for compatibility
            '-x265-params', 'log-level=error',
            str(out_file)
        ]
        
        subprocess.run(cmd_encode, capture_output=True, check=True)
        encode_time = time.time() - t0
        
        compressed_bytes = out_file.stat().st_size
        
        # Decode
        t0 = time.time()
        dec_file = output_path / "temp_decoded.yuv"
        
        cmd_decode = [
            'ffmpeg', '-y',
            '-i', str(out_file),
            '-pix_fmt', 'gray',
            '-f', 'rawvideo',
            str(dec_file)
        ]
        subprocess.run(cmd_decode, capture_output=True, check=True)
        
        # Read decoded frames
        with open(dec_file, 'rb') as f:
            raw_data = f.read()
        
        # Note: HEVC converts to YUV420, losing some spatial resolution in chroma
        # For grayscale, we only care about Y channel
        V_dec_8bit = np.frombuffer(raw_data, dtype=np.uint8)[:Z*Y*X].reshape(Z, Y, X)
        decode_time = time.time() - t0
        
        # Convert back to original scale
        V_dec_orig = (V_dec_8bit.astype(np.float64) / 255 * (v_max - v_min) + v_min).astype(volume.dtype)
        
        # Cleanup temp files
        raw_file.unlink(missing_ok=True)
        dec_file.unlink(missing_ok=True)
        
        # Evaluate
        psnr, ssim = evaluate_reconstruction(volume, V_dec_orig)
        
        result = add_result(
            name=f"H.265/HEVC (CRF={crf})",
            compressed_bytes=compressed_bytes,
            original_bytes=volume.nbytes,
            psnr=psnr, ssim=ssim,
            encode_time=encode_time,
            decode_time=decode_time,
            is_lossless=False,
            notes=f"libx265 preset=medium, {Z} frames as z-slices"
        )
        results_hevc.append(result)
    
    return results_hevc

# Run at multiple quality levels
try:
    codec_hevc_video(V_original, OUTPUT_DIR, crf_values=[18, 25, 32])
except Exception as e:
    print(f"HEVC codec error: {e}")

HEVC codec error: Command '['ffmpeg', '-y', '-f', 'rawvideo', '-pix_fmt', 'gray', '-s', '813x647', '-r', '1', '-i', 'codec_comparison_outputs/temp_raw.yuv', '-c:v', 'libx265', '-crf', '18', '-preset', 'medium', '-pix_fmt', 'yuv420p', '-x265-params', 'log-level=error', 'codec_comparison_outputs/volume_hevc_crf18.mp4']' returned non-zero exit status 183.


---
# Part C: Neural Implicit Baselines

Coordinate-based implicit neural representations (INR) with comparable optimization budgets.

## 9) Neural Baseline Utilities

In [15]:
def make_coord_grid_3d(shape_zyx: Tuple[int, int, int], device: str) -> torch.Tensor:
    """Create normalized coordinate grid in [-1, 1]³."""
    Z, Y, X = shape_zyx
    zs = torch.linspace(-1, 1, Z, device=device)
    ys = torch.linspace(-1, 1, Y, device=device)
    xs = torch.linspace(-1, 1, X, device=device)
    zz, yy, xx = torch.meshgrid(zs, ys, xs, indexing='ij')
    coords = torch.stack([xx, yy, zz], dim=-1)  # (Z, Y, X, 3)
    return coords

def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def model_size_bytes(model: nn.Module, quantize_bits: int = 32) -> int:
    """Estimate model size in bytes (with optional quantization)."""
    n_params = count_parameters(model)
    return n_params * (quantize_bits // 8)

def quantize_model_weights(model: nn.Module, bits: int = 8) -> bytes:
    """
    Quantize model weights and return compressed bytes.
    Uses per-tensor min-max quantization + gzip compression.
    """
    quantized_data = io.BytesIO()
    
    for name, param in model.named_parameters():
        data = param.detach().cpu().numpy()
        
        # Per-tensor min-max quantization
        p_min, p_max = data.min(), data.max()
        scale = (p_max - p_min) / (2**bits - 1) if p_max != p_min else 1.0
        
        # Quantize
        q_data = np.round((data - p_min) / scale).astype(np.uint8 if bits == 8 else np.uint16)
        
        # Store metadata + quantized values
        np.savez_compressed(quantized_data, 
                           data=q_data, 
                           min=np.array([p_min]),
                           scale=np.array([scale]),
                           shape=np.array(data.shape))
    
    # Apply gzip for additional compression
    raw_bytes = quantized_data.getvalue()
    compressed = gzip.compress(raw_bytes, compresslevel=9)
    return compressed

def save_quantized_model(model: nn.Module, path: Path, bits: int = 8) -> int:
    """Save quantized model weights and return file size."""
    compressed = quantize_model_weights(model, bits)
    with open(path, 'wb') as f:
        f.write(compressed)
    return len(compressed)

@torch.no_grad()
def reconstruct_from_inr(model: nn.Module, coords_grid: torch.Tensor, 
                         chunk_size: int = 500_000) -> torch.Tensor:
    """Reconstruct volume from INR model in chunks (memory efficient)."""
    model.eval()
    Z, Y, X, _ = coords_grid.shape
    coords_flat = coords_grid.reshape(-1, 3)
    
    output = torch.empty(coords_flat.shape[0], device=coords_flat.device)
    
    for i in range(0, coords_flat.shape[0], chunk_size):
        chunk = coords_flat[i:i+chunk_size]
        output[i:i+chunk_size] = model(chunk).squeeze(-1)
    
    return output.reshape(Z, Y, X).clamp(0, 1)

## 10) SIREN-3D (Sinusoidal INR)

In [16]:
class SineLayer(nn.Module):
    """
    Sine activation layer for SIREN networks.
    See: Sitzmann et al., "Implicit Neural Representations with Periodic Activation Functions"
    """
    def __init__(self, in_features: int, out_features: int, 
                 is_first: bool = False, omega_0: float = 30.0):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.linear = nn.Linear(in_features, out_features)
        self._init_weights()
    
    def _init_weights(self):
        with torch.no_grad():
            if self.is_first:
                # First layer: uniform in [-1/in, 1/in]
                bound = 1.0 / self.linear.in_features
            else:
                # Hidden layers: uniform in [-sqrt(6/in)/omega, sqrt(6/in)/omega]
                bound = np.sqrt(6.0 / self.linear.in_features) / self.omega_0
            self.linear.weight.uniform_(-bound, bound)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sin(self.omega_0 * self.linear(x))


class SIREN3D(nn.Module):
    """
    SIREN network for 3D implicit neural representation.
    Maps (x, y, z) coordinates to intensity values.
    """
    def __init__(self, in_features: int = 3, hidden_features: int = 256, 
                 hidden_layers: int = 4, out_features: int = 1, 
                 omega_0: float = 30.0, omega_hidden: float = 30.0):
        super().__init__()
        
        layers = []
        
        # First layer
        layers.append(SineLayer(in_features, hidden_features, is_first=True, omega_0=omega_0))
        
        # Hidden layers
        for _ in range(hidden_layers - 1):
            layers.append(SineLayer(hidden_features, hidden_features, omega_0=omega_hidden))
        
        # Output layer (linear, no sine)
        self.net = nn.Sequential(*layers)
        self.final = nn.Linear(hidden_features, out_features)
        
        # Initialize final layer
        with torch.no_grad():
            bound = np.sqrt(6.0 / hidden_features) / omega_hidden
            self.final.weight.uniform_(-bound, bound)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.final(self.net(x))


def train_siren(volume: torch.Tensor, coords_grid: torch.Tensor,
                hidden_features: int = 256, hidden_layers: int = 4,
                steps: int = 5000, batch_size: int = 100_000,
                lr: float = 1e-4, omega_0: float = 30.0) -> Tuple[SIREN3D, Dict]:
    """Train a SIREN model on the volume."""
    
    model = SIREN3D(
        in_features=3,
        hidden_features=hidden_features,
        hidden_layers=hidden_layers,
        out_features=1,
        omega_0=omega_0
    ).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)
    
    Z, Y, X, _ = coords_grid.shape
    total_voxels = Z * Y * X
    coords_flat = coords_grid.reshape(-1, 3)
    values_flat = volume.reshape(-1)
    
    losses = []
    
    pbar = tqdm(range(steps), desc="Training SIREN")
    for step in pbar:
        # Random batch sampling
        idx = torch.randint(0, total_voxels, (batch_size,), device=DEVICE)
        coords_batch = coords_flat[idx]
        target_batch = values_flat[idx]
        
        # Forward pass
        pred = model(coords_batch).squeeze(-1)
        loss = F.mse_loss(pred, target_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        losses.append(float(loss))
        
        if (step + 1) % 500 == 0:
            pbar.set_postfix({'loss': f'{loss:.6f}'})
    
    return model, {'losses': losses}


def codec_siren(volume_norm: np.ndarray, volume_orig: np.ndarray, output_path: Path,
                configs: List[Dict] = None) -> List[CompressionResult]:
    """
    SIREN-3D codec at different model sizes/quality levels.
    """
    if configs is None:
        configs = [
            {'hidden': 128, 'layers': 3, 'steps': 3000, 'name': 'Small'},
            {'hidden': 256, 'layers': 4, 'steps': 5000, 'name': 'Medium'},
            {'hidden': 384, 'layers': 5, 'steps': 8000, 'name': 'Large'},
        ]
    
    results_siren = []
    V_t = torch.from_numpy(volume_norm).to(DEVICE)
    coords = make_coord_grid_3d(volume_norm.shape, DEVICE)
    
    for cfg in configs:
        print(f"\nTraining SIREN-{cfg['name']} (hidden={cfg['hidden']}, layers={cfg['layers']})...")
        
        t0 = time.time()
        model, history = train_siren(
            V_t, coords,
            hidden_features=cfg['hidden'],
            hidden_layers=cfg['layers'],
            steps=cfg['steps']
        )
        encode_time = time.time() - t0
        
        # Save quantized model (8-bit)
        model_path = output_path / f"siren_{cfg['name'].lower()}.bin"
        compressed_bytes = save_quantized_model(model, model_path, bits=8)
        
        # Reconstruct
        t0 = time.time()
        V_rec = reconstruct_from_inr(model, coords)
        decode_time = time.time() - t0
        
        # Convert back to original scale
        v_min, v_max = float(volume_orig.min()), float(volume_orig.max())
        V_rec_np = (V_rec.cpu().numpy() * (v_max - v_min) + v_min).astype(volume_orig.dtype)
        
        # Evaluate
        psnr, ssim = evaluate_reconstruction(volume_orig, V_rec_np)
        
        n_params = count_parameters(model)
        result = add_result(
            name=f"SIREN-3D ({cfg['name']})",
            compressed_bytes=compressed_bytes,
            original_bytes=volume_orig.nbytes,
            psnr=psnr, ssim=ssim,
            encode_time=encode_time,
            decode_time=decode_time,
            is_lossless=False,
            notes=f"{n_params:,} params, 8-bit quantized + gzip"
        )
        results_siren.append(result)
    
    return results_siren

# Train SIREN models
siren_results = codec_siren(V_norm, V_original, OUTPUT_DIR)


Training SIREN-Small (hidden=128, layers=3)...


Training SIREN:   0%|          | 0/3000 [00:00<?, ?it/s]/tmp/ipykernel_17750/3388789649.py:101: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses.append(float(loss))
Training SIREN: 100%|██████████| 3000/3000 [00:12<00:00, 233.35it/s, loss=0.000445]



SIREN-3D (Small)
  Compressed: 29,874 bytes (0.03 MB)
  Ratio: 1760.77x | BPP: 0.0045
  PSNR: -4.88 dB | SSIM: 0.8929
  Encode: 13.73s | Decode: 0.63s
  Notes: 33,665 params, 8-bit quantized + gzip

Training SIREN-Medium (hidden=256, layers=4)...


Training SIREN: 100%|██████████| 5000/5000 [00:44<00:00, 113.11it/s, loss=0.000148]



SIREN-3D (Medium)
  Compressed: 168,761 bytes (0.17 MB)
  Ratio: 311.69x | BPP: 0.0257
  PSNR: -0.48 dB | SSIM: 0.9248
  Encode: 44.21s | Decode: 0.67s
  Notes: 198,657 params, 8-bit quantized + gzip

Training SIREN-Large (hidden=384, layers=5)...


Training SIREN: 100%|██████████| 8000/8000 [02:13<00:00, 59.75it/s, loss=0.000103]



SIREN-3D (Large)
  Compressed: 511,088 bytes (0.51 MB)
  Ratio: 102.92x | BPP: 0.0777
  PSNR: 1.19 dB | SSIM: 0.9331
  Encode: 133.90s | Decode: 1.49s
  Notes: 593,281 params, 8-bit quantized + gzip


## 11) PE-MLP (Fourier/Positional Encoding Features)

In [17]:
class PositionalEncoding(nn.Module):
    """
    Fourier/Positional Encoding for coordinate inputs.
    Maps low-dimensional coordinates to higher-dimensional space using sinusoidal functions.
    See: Mildenhall et al., "NeRF: Representing Scenes as Neural Radiance Fields"
    """
    def __init__(self, in_features: int = 3, num_frequencies: int = 10, 
                 include_input: bool = True):
        super().__init__()
        self.in_features = in_features
        self.num_frequencies = num_frequencies
        self.include_input = include_input
        
        # Create frequency bands: 2^0, 2^1, ..., 2^(L-1) * pi
        freq_bands = 2.0 ** torch.linspace(0, num_frequencies - 1, num_frequencies)
        self.register_buffer('freq_bands', freq_bands * np.pi)
        
        # Output dimension: input + 2 * num_frequencies * in_features (sin + cos for each freq)
        self.out_features = in_features * (1 + 2 * num_frequencies) if include_input else in_features * 2 * num_frequencies
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input coordinates of shape (..., in_features)
        Returns:
            Encoded coordinates of shape (..., out_features)
        """
        encoded = []
        
        if self.include_input:
            encoded.append(x)
        
        for freq in self.freq_bands:
            encoded.append(torch.sin(freq * x))
            encoded.append(torch.cos(freq * x))
        
        return torch.cat(encoded, dim=-1)


class PEMLP(nn.Module):
    """
    MLP with Positional Encoding (Fourier Features) for 3D implicit neural representation.
    Maps (x, y, z) coordinates through PE then through standard MLP to intensity values.
    """
    def __init__(self, in_features: int = 3, hidden_features: int = 256, 
                 hidden_layers: int = 4, out_features: int = 1,
                 num_frequencies: int = 10, skip_connection: bool = True):
        super().__init__()
        
        self.skip_connection = skip_connection
        self.skip_layer = hidden_layers // 2  # Skip connection at middle layer
        
        # Positional encoding
        self.pe = PositionalEncoding(in_features, num_frequencies, include_input=True)
        pe_dim = self.pe.out_features
        
        # Build MLP layers
        layers = []
        
        # First layer: PE output -> hidden
        layers.append(nn.Linear(pe_dim, hidden_features))
        layers.append(nn.ReLU(inplace=True))
        
        # Hidden layers
        for i in range(hidden_layers - 1):
            if skip_connection and i == self.skip_layer:
                # Skip connection: concatenate PE features
                layers.append(nn.Linear(hidden_features + pe_dim, hidden_features))
            else:
                layers.append(nn.Linear(hidden_features, hidden_features))
            layers.append(nn.ReLU(inplace=True))
        
        self.layers = nn.ModuleList(layers)
        
        # Output layer
        self.final = nn.Linear(hidden_features, out_features)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Apply positional encoding
        pe_x = self.pe(x)
        h = pe_x
        
        layer_idx = 0
        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.Linear):
                if self.skip_connection and layer_idx == self.skip_layer + 1:
                    # Concatenate PE features for skip connection
                    h = torch.cat([h, pe_x], dim=-1)
                h = layer(h)
                layer_idx += 1
            else:
                h = layer(h)
        
        return self.final(h)


def train_pemlp(volume: torch.Tensor, coords_grid: torch.Tensor,
                hidden_features: int = 256, hidden_layers: int = 4,
                num_frequencies: int = 10, steps: int = 5000, 
                batch_size: int = 100_000, lr: float = 1e-3) -> Tuple[PEMLP, Dict]:
    """Train a PE-MLP model on the volume."""
    
    model = PEMLP(
        in_features=3,
        hidden_features=hidden_features,
        hidden_layers=hidden_layers,
        out_features=1,
        num_frequencies=num_frequencies,
        skip_connection=True
    ).to(DEVICE)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)
    
    Z, Y, X, _ = coords_grid.shape
    total_voxels = Z * Y * X
    coords_flat = coords_grid.reshape(-1, 3)
    values_flat = volume.reshape(-1)
    
    losses = []
    
    pbar = tqdm(range(steps), desc="Training PE-MLP")
    for step in pbar:
        # Random batch sampling
        idx = torch.randint(0, total_voxels, (batch_size,), device=DEVICE)
        coords_batch = coords_flat[idx]
        target_batch = values_flat[idx]
        
        # Forward pass
        pred = model(coords_batch).squeeze(-1)
        loss = F.mse_loss(pred, target_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        losses.append(float(loss))
        
        if (step + 1) % 500 == 0:
            pbar.set_postfix({'loss': f'{loss:.6f}'})
    
    return model, {'losses': losses}


def codec_pemlp(volume_norm: np.ndarray, volume_orig: np.ndarray, output_path: Path,
                configs: List[Dict] = None) -> List[CompressionResult]:
    """
    PE-MLP codec at different model sizes/quality levels.
    """
    if configs is None:
        configs = [
            {'hidden': 128, 'layers': 3, 'freqs': 8, 'steps': 3000, 'name': 'Small'},
            {'hidden': 256, 'layers': 4, 'freqs': 10, 'steps': 5000, 'name': 'Medium'},
            {'hidden': 384, 'layers': 5, 'freqs': 12, 'steps': 8000, 'name': 'Large'},
        ]
    
    results_pemlp = []
    V_t = torch.from_numpy(volume_norm).to(DEVICE)
    coords = make_coord_grid_3d(volume_norm.shape, DEVICE)
    
    for cfg in configs:
        print(f"\nTraining PE-MLP-{cfg['name']} (hidden={cfg['hidden']}, layers={cfg['layers']}, freqs={cfg['freqs']})...")
        
        t0 = time.time()
        model, history = train_pemlp(
            V_t, coords,
            hidden_features=cfg['hidden'],
            hidden_layers=cfg['layers'],
            num_frequencies=cfg['freqs'],
            steps=cfg['steps']
        )
        encode_time = time.time() - t0
        
        # Save quantized model (8-bit)
        model_path = output_path / f"pemlp_{cfg['name'].lower()}.bin"
        compressed_bytes = save_quantized_model(model, model_path, bits=8)
        
        # Reconstruct
        t0 = time.time()
        V_rec = reconstruct_from_inr(model, coords)
        decode_time = time.time() - t0
        
        # Convert back to original scale
        v_min, v_max = float(volume_orig.min()), float(volume_orig.max())
        V_rec_np = (V_rec.cpu().numpy() * (v_max - v_min) + v_min).astype(volume_orig.dtype)
        
        # Evaluate
        psnr, ssim = evaluate_reconstruction(volume_orig, V_rec_np)
        
        n_params = count_parameters(model)
        result = add_result(
            name=f"PE-MLP ({cfg['name']})",
            compressed_bytes=compressed_bytes,
            original_bytes=volume_orig.nbytes,
            psnr=psnr, ssim=ssim,
            encode_time=encode_time,
            decode_time=decode_time,
            is_lossless=False,
            notes=f"{n_params:,} params, {cfg['freqs']} freq bands, 8-bit quantized + gzip"
        )
        results_pemlp.append(result)
    
    return results_pemlp

# Train PE-MLP models
pemlp_results = codec_pemlp(V_norm, V_original, OUTPUT_DIR)


Training PE-MLP-Small (hidden=128, layers=3, freqs=8)...


Training PE-MLP:   2%|▎         | 75/3000 [00:00<00:07, 376.73it/s]

Training PE-MLP: 100%|██████████| 3000/3000 [00:11<00:00, 256.69it/s, loss=0.002237]



PE-MLP (Small)
  Compressed: 37,824 bytes (0.04 MB)
  Ratio: 1390.68x | BPP: 0.0058
  PSNR: -11.90 dB | SSIM: 0.6904
  Encode: 11.70s | Decode: 0.66s
  Notes: 46,337 params, 8 freq bands, 8-bit quantized + gzip

Training PE-MLP-Medium (hidden=256, layers=4, freqs=10)...


Training PE-MLP: 100%|██████████| 5000/5000 [00:34<00:00, 142.96it/s, loss=0.000276]



PE-MLP (Medium)
  Compressed: 170,276 bytes (0.17 MB)
  Ratio: 308.92x | BPP: 0.0259
  PSNR: -2.89 dB | SSIM: 0.8985
  Encode: 34.98s | Decode: 1.25s
  Notes: 230,145 params, 10 freq bands, 8-bit quantized + gzip

Training PE-MLP-Large (hidden=384, layers=5, freqs=12)...


Training PE-MLP: 100%|██████████| 8000/8000 [01:42<00:00, 78.42it/s, loss=0.000153]



PE-MLP (Large)
  Compressed: 437,152 bytes (0.44 MB)
  Ratio: 120.33x | BPP: 0.0665
  PSNR: -0.49 dB | SSIM: 0.9246
  Encode: 102.03s | Decode: 2.24s
  Notes: 649,729 params, 12 freq bands, 8-bit quantized + gzip


---
# Part D: NeuroGS-Codec (Ours)

Sparse mixture of anisotropic 3D Gaussians optimized via rate-distortion training.

## 12) NeuroGS-Codec (Ours)

In [ ]:
# NeuroGS-Codec: Gaussian Mixture Volume representation
# Load from pretrained checkpoint

def quat_to_rotmat(q: torch.Tensor) -> torch.Tensor:
    """Convert unit quaternion q=(w,x,y,z) into a rotation matrix R (3x3)."""
    w, x, y, z = q.unbind(-1)
    ww, xx, yy, zz = w*w, x*x, y*y, z*z
    wx, wy, wz = w*x, w*y, w*z
    xy, xz, yz = x*y, x*z, y*z
    R = torch.stack([
        ww+xx-yy-zz, 2*(xy-wz),     2*(xz+wy),
        2*(xy+wz),   ww-xx+yy-zz,   2*(yz-wx),
        2*(xz-wy),   2*(yz+wx),     ww-xx-yy+zz
    ], dim=-1).reshape(q.shape[:-1] + (3, 3))
    return R


def safe_normalize(q: torch.Tensor, eps=1e-8) -> torch.Tensor:
    """Normalize quaternions safely."""
    return q / (q.norm(dim=-1, keepdim=True) + eps)


class GaussianMixtureVolume(nn.Module):
    """
    Gaussian mixture field for NeuroGS-Codec:
      f(x) = sum_i a_i * exp( -1/2 * || (R_i^T (x - mu_i)) / s_i ||^2 ) + b
    
    Parameters per Gaussian i:
      mu_i   : mean / center in normalized coordinates [-1,1]^3
      s_i    : axis-aligned scale (std dev) stored as log_s
      q_i    : quaternion for rotation R_i
      a_i    : amplitude (intensity contribution)
    Global:
      b      : global bias / background level
    """
    def __init__(self, N: int, device: str = DEVICE):
        super().__init__()
        self.N = N
        # Gaussian centers mu_i (N,3)
        self.mu = nn.Parameter(torch.zeros(N, 3, device=device))
        # log-scales (N,3); scale s = exp(log_s)
        self.log_s = nn.Parameter(torch.zeros(N, 3, device=device) - 2.0)
        # Quaternion rotations q_i (N,4) initialized to identity
        q = torch.zeros(N, 4, device=device)
        q[:, 0] = 1.0
        self.q = nn.Parameter(q)
        # Amplitudes (N,)
        self.a = nn.Parameter(torch.zeros(N, device=device))
        # Background bias (scalar)
        self.b = nn.Parameter(torch.tensor(0.0, device=device))
        self.sigma_cutoff = 3.0

    def forward_chunked(self, x: torch.Tensor, gaussian_chunk_size: int = 10000) -> torch.Tensor:
        """
        Memory-efficient forward pass that chunks over Gaussians.
        For large N, this avoids creating huge (P, N, 3) tensors.
        """
        P = x.shape[0]
        N = self.N
        
        # Pre-compute rotation matrices and scales once
        s = torch.exp(self.log_s).clamp(1e-4, 10.0)                # (N,3)
        qn = safe_normalize(self.q)                                 # (N,4)
        R = quat_to_rotmat(qn)                                      # (N,3,3)
        Rt = R.transpose(-1, -2)                                    # (N,3,3)
        
        # Accumulate contributions from Gaussian chunks
        pred = torch.zeros(P, device=x.device, dtype=x.dtype)
        
        for g_start in range(0, N, gaussian_chunk_size):
            g_end = min(g_start + gaussian_chunk_size, N)
            
            # Get chunk of Gaussian parameters
            mu_chunk = self.mu[g_start:g_end]           # (Nc, 3)
            s_chunk = s[g_start:g_end]                  # (Nc, 3)
            Rt_chunk = Rt[g_start:g_end]                # (Nc, 3, 3)
            a_chunk = self.a[g_start:g_end]             # (Nc,)
            
            # Compute displacement: (P, Nc, 3)
            dx = x[:, None, :] - mu_chunk[None, :, :]
            
            # Transform to local frame
            y = torch.einsum("pni,nij->pnj", dx, Rt_chunk)  # (P, Nc, 3)
            y = y / (s_chunk[None, :, :] + 1e-8)
            
            # Gaussian kernel
            exp_term = -0.5 * (y * y).sum(dim=-1)           # (P, Nc)
            g = torch.exp(exp_term)
            
            # Accumulate weighted sum
            pred += (g * a_chunk[None, :]).sum(dim=1)
        
        # Add bias
        pred = pred + self.b
        return pred

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Evaluate the mixture field at query points x of shape (P,3)."""
        # Use chunked forward for large N to avoid OOM
        if self.N > 20000:
            return self.forward_chunked(x, gaussian_chunk_size=10000)
        
        mu = self.mu[None, :, :]            # (1,N,3)
        dx = x[:, None, :] - mu             # (P,N,3)

        s = torch.exp(self.log_s).clamp(1e-4, 10.0)                # (N,3)
        qn = safe_normalize(self.q)                                 # (N,4)
        R = quat_to_rotmat(qn)                                      # (N,3,3)
        Rt = R.transpose(-1, -2)

        y = torch.einsum("pni,nij->pnj", dx, Rt)                    # (P,N,3)
        y = y / (s[None, :, :] + 1e-8)

        exp_term = -0.5 * (y * y).sum(dim=-1)                      # (P,N)
        g = torch.exp(exp_term)

        pred = (g * self.a[None, :]).sum(dim=1) + self.b
        return pred

    @classmethod
    def from_checkpoint(cls, ckpt_path: str, device: str = DEVICE) -> 'GaussianMixtureVolume':
        """Load model from checkpoint."""
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        
        # Get model state dict
        state = ckpt.get('model_state', ckpt.get('state_dict', ckpt))
        
        # Determine N from mu shape
        mu = state['mu']
        N = mu.shape[0]
        
        model = cls(N, device=device)
        model.load_state_dict(state)
        return model


@torch.no_grad()
def reconstruct_from_neurogs(model: GaussianMixtureVolume, coords_grid: torch.Tensor,
                              chunk_size: int = 50000) -> torch.Tensor:
    """
    Reconstruct volume from NeuroGS model in chunks (memory efficient).
    Uses smaller chunk size to work with the chunked Gaussian evaluation.
    """
    model.eval()
    Z, Y, X, _ = coords_grid.shape
    coords_flat = coords_grid.reshape(-1, 3)
    
    output = torch.empty(coords_flat.shape[0], device=coords_flat.device)
    
    for i in tqdm(range(0, coords_flat.shape[0], chunk_size), desc="Reconstructing"):
        chunk = coords_flat[i:i+chunk_size]
        output[i:i+chunk_size] = model(chunk)
    
    return output.reshape(Z, Y, X)


def get_neurogs_compressed_size(ckpt_path: str, bits: int = 8) -> int:
    """
    Calculate the compressed size of NeuroGS model using 8-bit quantization + gzip.
    Similar to how we quantize SIREN/PE-MLP for fair comparison.
    """
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    state = ckpt.get('model_state', ckpt.get('state_dict', ckpt))
    
    quantized_data = io.BytesIO()
    
    for name, param in state.items():
        if isinstance(param, torch.Tensor):
            data = param.numpy()
            
            # Per-tensor min-max quantization
            p_min, p_max = data.min(), data.max()
            scale = (p_max - p_min) / (2**bits - 1) if p_max != p_min else 1.0
            
            # Quantize
            q_data = np.round((data - p_min) / scale).astype(np.uint8 if bits == 8 else np.uint16)
            
            # Store metadata + quantized values
            np.savez_compressed(quantized_data, 
                               data=q_data, 
                               min=np.array([p_min]),
                               scale=np.array([scale]),
                               shape=np.array(data.shape))
    
    # Apply gzip for additional compression
    raw_bytes = quantized_data.getvalue()
    compressed = gzip.compress(raw_bytes, compresslevel=9)
    return len(compressed)


def codec_neurogs(volume_norm: np.ndarray, volume_orig: np.ndarray, output_path: Path,
                  checkpoint_path: str = None) -> CompressionResult:
    """
    NeuroGS-Codec evaluation from a pretrained checkpoint.
    """
    # Default to final checkpoint if not specified
    if checkpoint_path is None:
        checkpoint_path = "checkpoints/neurogs_codec_ckpt_final.pt"
    
    if not os.path.exists(checkpoint_path):
        print(f"Skipping NeuroGS: checkpoint not found at {checkpoint_path}")
        return None
    
    print(f"\nLoading NeuroGS-Codec from {checkpoint_path}...")
    
    # Load model
    t0 = time.time()
    model = GaussianMixtureVolume.from_checkpoint(checkpoint_path, device=DEVICE)
    load_time = time.time() - t0
    
    # Get number of Gaussians
    N = model.N
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Loaded {N:,} Gaussians ({n_params:,} parameters)")
    
    # Calculate compressed size (8-bit quantized + gzip, same as other neural methods)
    compressed_bytes = get_neurogs_compressed_size(checkpoint_path, bits=8)
    
    # Build coordinate grid
    coords = make_coord_grid_3d(volume_norm.shape, DEVICE)
    
    # Reconstruct
    t0 = time.time()
    V_rec = reconstruct_from_neurogs(model, coords)
    decode_time = time.time() - t0
    
    # Debug: Check model output range
    rec_min, rec_max = float(V_rec.min()), float(V_rec.max())
    print(f"  Model output range: [{rec_min:.4f}, {rec_max:.4f}]")
    print(f"  Target normalized range: [0, 1]")
    
    # Normalize the reconstruction to [0,1] based on its own range
    # This handles cases where model output is not exactly in [0,1]
    V_rec_normalized = (V_rec - rec_min) / (rec_max - rec_min + 1e-8)
    V_rec_normalized = V_rec_normalized.clamp(0, 1)
    
    # Convert back to original scale
    v_min, v_max = float(volume_orig.min()), float(volume_orig.max())
    V_rec_np = (V_rec_normalized.cpu().numpy() * (v_max - v_min) + v_min).astype(volume_orig.dtype)
    
    print(f"  Original range: [{v_min}, {v_max}]")
    print(f"  Reconstructed range: [{V_rec_np.min()}, {V_rec_np.max()}]")
    
    # Evaluate
    psnr, ssim = evaluate_reconstruction(volume_orig, V_rec_np)
    
    # For NeuroGS, encode_time would be the training time (already done)
    # We report load_time as a proxy, but note this isn't the full encode time
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    train_iterations = ckpt.get('iteration', 'N/A')
    
    result = add_result(
        name="NeuroGS-Codec (Ours)",
        compressed_bytes=compressed_bytes,
        original_bytes=volume_orig.nbytes,
        psnr=psnr, ssim=ssim,
        encode_time=0.0,  # Training time not measured here
        decode_time=decode_time,
        is_lossless=False,
        notes=f"{N:,} Gaussians, {n_params:,} params, iter={train_iterations}, 8-bit quantized + gzip"
    )
    
    return result


# Evaluate NeuroGS-Codec
neurogs_result = codec_neurogs(V_norm, V_original, OUTPUT_DIR)


Loading NeuroGS-Codec from checkpoints/neurogs_codec_ckpt_final.pt...
  Loaded 169,295 Gaussians (1,862,246 parameters)


Reconstructing: 100%|██████████| 1053/1053 [48:56<00:00,  2.79s/it]



NeuroGS-Codec (Ours)
  Compressed: 1,236,796 bytes (1.24 MB)
  Ratio: 42.53x | BPP: 0.1881
  PSNR: -2.19 dB | SSIM: 0.9194
  Encode: 0.00s | Decode: 2936.39s
  Notes: 169,295 Gaussians, 1,862,246 params, iter=30000, 8-bit quantized + gzip
